<a href="https://colab.research.google.com/github/prometricas/Peajes_Laura_Toro/blob/main/Peajes_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Pronóstico econométrico de Recaudo para flujo vehicular en peajes**
* Autora: Laura Maria Toro Montoya
* Email: lauratoromont28@gmail.com




---
# **1. Limpieza: revisión de continuidad e imputación de faltantes**
---
### Propósito de esta fase
En esta etapa preparo la serie diaria para que quede **continua, ordenada y coherente** antes de construir variables exógenas o ajustar modelos de pronóstico.

El objetivo principal es:

- verificar que la frecuencia sea **diaria**,
- identificar fechas faltantes,
- completar esos huecos con una regla **simple, explicable y consistente con la estacionalidad semanal**,
- dejar una base limpia para las fases siguientes.

---

### Revisión de continuidad de la serie
Primero comparo la base original contra un calendario diario completo entre la fecha mínima y la fecha máxima observada.  
Con esa validación aparecen **tres fechas faltantes**:

- **2022-07-28**
- **2022-08-06**
- **2022-11-17**

Estas ausencias son **aisladas**, por lo que no se requiere una estrategia compleja de reconstrucción.

> **Criterio adoptado:** en lugar de usar una interpolación lineal simple entre días consecutivos, completo cada hueco con una **imputación estacional de 7 días**, porque la serie presenta un patrón semanal muy marcado.

---

### Regla de imputación aplicada
Para cada fecha faltante \( t \), estimo el valor a partir del mismo día de la semana anterior y del mismo día de la semana siguiente.

_Ecuación para el recaudo:_

$$
\hat{R}_t = \frac{R_{t-7} + R_{t+7}}{2}
$$

_Ecuación para los movimientos_:

$$
\hat{M}_t = \frac{M_{t-7} + M_{t+7}}{2}
$$


---

### Justificación metodológica
Esta decisión se toma porque:

**1.** La serie tiene una **estacionalidad semanal fuerte**.  
**2.** Las fechas faltantes no forman bloques largos, sino huecos puntuales.  
**3.** La imputación con \( t-7 \) y \( t+7 \) conserva mejor el comportamiento típico del mismo día de la semana.  
**4.** El procedimiento es **fácil de explicar en una exposición** y deja una trazabilidad clara.

> En términos prácticos, esta regla evita que un jueves o un sábado faltante quede arrastrado por valores atípicos del día inmediatamente anterior o posterior.

---

### Resultado esperado de esta fase
Al finalizar esta etapa, la base debe quedar con:

- una secuencia diaria completa,
- **cero valores faltantes** en las variables principales,
- una marca de control para identificar qué registros fueron imputados,
- gráficos que permitan mostrar el **antes y el después** de la imputación.

---

### Variables tratadas en esta fase
En esta limpieza inicial se corrigen las dos variables base del problema:

| Variable | Descripción |
|---|---|
| **recaudo** | Valor diario recaudado en peajes |
| **movimientos** | Número diario de movimientos registrados |

---

In [ ]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
plt.style.use("seaborn-v0_8-whitegrid")

# ------------------------------------------------------------
# 1. Carga de la base
# ------------------------------------------------------------

# Si hace falta subirlo manualmente, se puede usar esto:
from google.colab import files
files.upload()
ruta_excel = "/content/data_peajes.xlsx"

# Cargo la base original
df = pd.read_excel(ruta_excel)

# Estandarizo nombres para evitar errores por tildes o espacios
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.normalize("NFKD")
      .str.encode("ascii", errors="ignore")
      .str.decode("utf-8")
)

# Convierto fecha y ordeno la serie
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)

print("Dimensión original:", df.shape)
print("\nColumnas detectadas:", df.columns.tolist())
print("\nPrimeras filas:")
display(df.head())

# ------------------------------------------------------------
# 2. Revisión de continuidad diaria
# ------------------------------------------------------------

# Armo un calendario diario completo desde la primera hasta la última fecha
calendario = pd.DataFrame({
    "fecha": pd.date_range(df["fecha"].min(), df["fecha"].max(), freq="D")
})

# Uno la base contra ese calendario para detectar huecos
base = calendario.merge(df, on="fecha", how="left")

# Marco las filas que quedaron vacías por ausencia en la fuente
base["es_imputado"] = base["recaudo"].isna()

faltantes = base.loc[base["es_imputado"], "fecha"].copy()

print("\nFechas faltantes detectadas:")
print(faltantes.dt.strftime("%Y-%m-%d").tolist())

# ------------------------------------------------------------
# 3. Completo el día de la semana desde la fecha
# ------------------------------------------------------------

dias_es = {
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miercoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sabado",
    "Sunday": "Domingo"
}

base["dia_semana"] = base["fecha"].dt.day_name().map(dias_es)

# ------------------------------------------------------------
# 4. Imputación simple y fácil de explicar
# ------------------------------------------------------------

# Tomo la fecha como índice para usar desplazamientos de 7 días
base = base.set_index("fecha")

# Para cada fecha faltante:
# imputo con el promedio del mismo día de la semana
# de la semana anterior y la siguiente
for fecha in faltantes:
    for col in ["recaudo", "movimientos"]:
        valor_semana_anterior = base.loc[fecha - pd.Timedelta(days=7), col]
        valor_semana_siguiente = base.loc[fecha + pd.Timedelta(days=7), col]
        base.loc[fecha, col] = (valor_semana_anterior + valor_semana_siguiente) / 2

base = base.reset_index()

# Redondeo porque ambas variables deben quedar enteras
base["recaudo"] = base["recaudo"].round(0).astype("int64")
base["movimientos"] = base["movimientos"].round(0).astype("int64")

print("\nValores imputados:")
display(base.loc[base["es_imputado"], ["fecha", "dia_semana", "recaudo", "movimientos"]])

# ------------------------------------------------------------
# 5. Validaciones rápidas
# ------------------------------------------------------------

print("Fechas duplicadas:", base["fecha"].duplicated().sum())
print("Nulos en fecha:", base["fecha"].isna().sum())
print("Nulos en recaudo:", base["recaudo"].isna().sum())
print("Nulos en movimientos:", base["movimientos"].isna().sum())

print("\nSaltos entre fechas después de limpiar:")
display(base["fecha"].diff().dropna().value_counts().to_frame("conteo"))

# ------------------------------------------------------------
# 6. Gráfico general con las fechas imputadas
# ------------------------------------------------------------

def formato_millones(x, pos):
    return f"${x/1e6:,.0f}M".replace(",", ".")

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(base["fecha"], base["recaudo"], linewidth=1.4, label="Serie diaria")
ax.scatter(
    base.loc[base["es_imputado"], "fecha"],
    base.loc[base["es_imputado"], "recaudo"],
    s=90,
    color="crimson",
    edgecolor="black",
    zorder=5,
    label="Dato imputado"
)

ax.set_title("Recaudo diario con fechas imputadas", fontsize=15, weight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("Recaudo")
ax.yaxis.set_major_formatter(FuncFormatter(formato_millones))
ax.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. Zoom de cada imputación para exposición
# ------------------------------------------------------------

fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharey=False)

for ax, fecha in zip(axes, faltantes):
    ventana = base[
        (base["fecha"] >= fecha - pd.Timedelta(days=10)) &
        (base["fecha"] <= fecha + pd.Timedelta(days=10))
    ].copy()

    # Recreo la serie con el hueco original para mostrar el antes
    ventana_hueco = ventana.copy()
    ventana_hueco.loc[ventana_hueco["fecha"] == fecha, "recaudo"] = np.nan

    ax.plot(
        ventana_hueco["fecha"],
        ventana_hueco["recaudo"],
        marker="o",
        linewidth=1.5,
        label="Serie original con hueco"
    )

    ax.plot(
        ventana["fecha"],
        ventana["recaudo"],
        linewidth=2.2,
        alpha=0.90,
        label="Serie después de imputar"
    )

    punto = ventana.loc[ventana["fecha"] == fecha]

    ax.scatter(
        punto["fecha"],
        punto["recaudo"],
        s=120,
        color="crimson",
        edgecolor="black",
        zorder=6,
        label="Valor imputado"
    )

    ax.axvline(fecha, color="gray", linestyle="--", alpha=0.7)
    ax.set_title(f"Ventana alrededor de {fecha.strftime('%Y-%m-%d')}", fontsize=12, weight="bold")
    ax.set_ylabel("Recaudo")
    ax.yaxis.set_major_formatter(FuncFormatter(formato_millones))
    ax.legend(loc="upper left")

axes[-1].set_xlabel("Fecha")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. Guardo la base limpia de Fase 1
# ------------------------------------------------------------

base = base.sort_values("fecha").reset_index(drop=True)

ruta_salida = "/content/data_peajes_fase1.xlsx"
base.to_excel(ruta_salida, index=False)

print(f"\nArchivo limpio guardado en: {ruta_salida}")

# **2. Construcción de variables explicativas**
En esa siguiente etapa se incorporarán potenciales variables explicativas y covariables relevantes para mejorar el pronóstico.

## **2.1. Combustible**

In [7]:
import re
import unicodedata
import numpy as np
import pandas as pd
import requests

# Trabajo sobre la base que ya quedó en memoria desde la Fase 1
base = base.copy()
base["fecha"] = pd.to_datetime(base["fecha"])

# Descargo la API
url_api = "https://www.datos.gov.co/resource/gjy9-tpph.json?$limit=5000"
respuesta = requests.get(url_api, timeout=60)
respuesta.raise_for_status()
df_api = pd.DataFrame(respuesta.json())

# Normalizo nombres de columnas
def normalizar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    return texto.strip("_")

df_api.columns = [normalizar_texto(c) for c in df_api.columns]

# Dejo solo las columnas útiles que sí existen en esta API
columnas_necesarias = ["periodo", "mes", "municipio", "producto", "precio"]
faltantes = [c for c in columnas_necesarias if c not in df_api.columns]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias en la API: {faltantes}")

df_comb = df_api[columnas_necesarias].copy()

# Limpio tipos y texto
df_comb["municipio"] = df_comb["municipio"].astype(str).str.upper().str.strip()
df_comb["producto"] = df_comb["producto"].astype(str).str.upper().str.strip()
df_comb["mes"] = df_comb["mes"].astype(str).str.upper().str.strip()
df_comb["precio"] = pd.to_numeric(df_comb["precio"], errors="coerce")

# Filtro Medellín
df_comb = df_comb[df_comb["municipio"].str.contains("MEDELLIN", na=False)].copy()

# Clasifico las dos series que voy a usar
def clasificar_producto(producto):
    if "GASOLINA CORRIENTE" in producto:
        return "gasolina_corriente_medellin"
    if "ACPM" in producto or "DIESEL" in producto or "BIODIESEL" in producto:
        return "acpm_medellin"
    return np.nan

df_comb["serie"] = df_comb["producto"].apply(clasificar_producto)
df_comb = df_comb[df_comb["serie"].notna()].copy()

# Construyo la fecha mensual
# Primero intento leer "periodo" directamente como fecha
df_comb["fecha_mes"] = pd.to_datetime(df_comb["periodo"], errors="coerce")

# Si "periodo" no viene en formato fecha, reconstruyo con año + mes
meses_es = {
    "ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6,
    "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "SETIEMBRE": 9, "OCTUBRE": 10,
    "NOVIEMBRE": 11, "DICIEMBRE": 12
}

mask_fecha_nula = df_comb["fecha_mes"].isna()

if mask_fecha_nula.any():
    df_aux = df_comb.loc[mask_fecha_nula, ["periodo", "mes"]].copy()
    df_aux["ano"] = df_aux["periodo"].astype(str).str.extract(r"(\d{4})")
    df_aux["mes_num"] = df_aux["mes"].map(meses_es)

    df_comb.loc[mask_fecha_nula, "fecha_mes"] = pd.to_datetime(
        dict(
            year=pd.to_numeric(df_aux["ano"], errors="coerce"),
            month=pd.to_numeric(df_aux["mes_num"], errors="coerce"),
            day=1
        ),
        errors="coerce"
    ).values

# Elimino filas sin fecha o sin precio
df_comb = df_comb.dropna(subset=["fecha_mes", "precio"]).copy()

# Consolido a mensual
df_comb_med_mensual = (
    df_comb.groupby(["fecha_mes", "serie"], as_index=False)["precio"]
    .mean()
    .pivot(index="fecha_mes", columns="serie", values="precio")
    .reset_index()
    .sort_values("fecha_mes")
)

df_comb_med_mensual.columns.name = None

# Expando a diaria usando el rango de la base principal
df_comb_med_diario = pd.DataFrame({
    "fecha": pd.date_range(base["fecha"].min(), base["fecha"].max(), freq="D")
})

df_comb_med_diario["fecha_mes"] = df_comb_med_diario["fecha"].dt.to_period("M").dt.to_timestamp()
df_comb_med_diario = df_comb_med_diario.merge(df_comb_med_mensual, on="fecha_mes", how="left")
df_comb_med_diario = df_comb_med_diario.drop(columns="fecha_mes")

# Uno combustibles a la base principal
base_modelo = base.merge(df_comb_med_diario, on="fecha", how="left")

# Muestro hallazgos
print("Productos útiles encontrados en Medellín:")
display(pd.DataFrame(sorted(df_comb["producto"].unique()), columns=["producto"]))

print("Tabla mensual de combustibles:")
display(df_comb_med_mensual.head(12))

print("Tabla diaria expandida:")
display(df_comb_med_diario.head(10))

print("Base principal unida con combustibles:")
display(base_modelo.head(10))

Productos útiles encontrados en Medellín:


,producto


Tabla mensual de combustibles:


,fecha_mes


Tabla diaria expandida:


,fecha
0,2021-01-01
1,2021-01-02
2,2021-01-03
3,2021-01-04
4,2021-01-05
5,2021-01-06
6,2021-01-07
7,2021-01-08
8,2021-01-09
9,2021-01-10


Base principal unida con combustibles:


,fecha,dia_semana,recaudo,movimientos,es_imputado
0,2021-01-01,Viernes,9695800,669,False
1,2021-01-02,Sabado,31887100,1814,False
2,2021-01-03,Domingo,94734100,7500,False
3,2021-01-04,Lunes,134606100,8391,False
4,2021-01-05,Martes,144778800,8703,False
5,2021-01-06,Miercoles,144530900,8634,False
6,2021-01-07,Jueves,134857700,8227,False
7,2021-01-08,Viernes,155547900,9886,False
8,2021-01-09,Sabado,51918900,2709,False
9,2021-01-10,Domingo,29106700,1511,False
